Training for deep learning reconstruction of 3D Kooshball GRE data.


In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

import sys

import gc
import time
import random

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.distributed as dist
import zarr as z

from juart.conopt.functional.fourier import (
    fourier_transform_adjoint,
    fourier_transform_forward,
    nonuniform_fourier_transform_adjoint,
)
from juart.conopt.tfs.fourier import nonuniform_transfer_function
from juart.dl.checkpoint.manager import CheckpointManager
from juart.dl.loss.loss import JointLoss
from juart.dl.model.unrollnet import (
    LookaheadModel,
    UnrolledNet,
)
from juart.dl.operation.modules import training, validation
from juart.dl.utils.dist import GradientAccumulator
from juart.vis.interactive import InteractiveFigure3D

## Definition of the most important variables

In [2]:
torch.autograd.set_detect_anomaly(True)

# dataset options
nX, nY, nZ = 128, 128, 128  # Number of pixels in x-/y-/z-direction
kspace_cutoff = False  # defines if nX,... should be reduced
nTI, nTE = 1, 1  # Number of measurements during the T1/T2 decay
shape = (nX, nY, nZ, nTI, nTE)  # Defining the shape later used for the model
nD = 1  # Number of subjects
nP = 1  # Number of slices per subject

# device options
device = "cuda:2"  # defines whether the model should be trained on the cpu or gpu
group = None
group_rank = 0
group_index = 0
num_groups = 1

# Training loop options
num_epochs = 30  # Number of epochs of the training
model_training = True  # Activate Training mode
model_validation = True  # Activate validation mode
save_checkpoint = (
    True  # Create save files that contain the current state of the training
)
checkpoint_frequency = 5  # Sets the number of iterations between creating save files
single_epoch = False  # Create a seperate save file after every single epoch
batch_size = 1  # Number of slices that should be used for training per batch

# model options
regularizer = "UNet"
cgiter = 100
num_unroll_blocks = 5
activation = "ReLU"
features = [8,16,32]
disable_progress_bar = False
kernel_size = (6, 6, 6)
Checkpoints = True

# CheckpointManager Options
load_model_state = True  # Load the last saved model state
load_averaged_model_state = True  # Load the last saved averaged model state
load_optim_state = True  # Load the las saved optimizer state
load_metrics = True  # Load the last saved metrics (iterations and loss)
directory = (
    f"model"  # Name that is used for the save directory of the model
)
root_dir = f"/home/jovyan/models/{regularizer}_0i_{cgiter}DC_{nP}P_masksplitR8"  # path of the model directory
backend = "local"  # backend of the model directory

batch_size_local = batch_size // num_groups
num_iterations = nD * nP * num_epochs  # complete number of iterations
iteration = 0  # current iteration number

dtype = torch.complex64

## Initializing worker groups

In [3]:
dist.init_process_group(
    backend="gloo", init_method="tcp://127.0.0.1:12345", world_size=1, rank=0
)

[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0


## Initializing the neural network
### Initializing the model
The model is the core of the neural network. Its forward pass describes the main steps of the neural network and the models features property describes the number of hidden features of the neural network and therefore its complexity. Furthermore the activation function and the number of iterations in dataconsistency term and regularization term is defined here.

In [4]:
model = UnrolledNet(
    shape,
    features=features,
    CG_Iter=cgiter,
    num_unroll_blocks=num_unroll_blocks,
    num_of_resblocks = 0,
    scale_factor = 0,
    activation=activation,
    disable_progress_bar=disable_progress_bar,
    pad_to = 0,
    timing_level=0,
    validation_level=0,
    kernel_size=kernel_size,
    regularizer=regularizer,
    Checkpoints = Checkpoints,
    device=device,
    dtype = dtype
)

### Initializing the loss function
The loss function defines how the difference between model prediction and correct result is calculated and it divides the loss into separate kinds of losses:
- kspace
- ispace
- wavelet
- hankel
- casorati

In [5]:
loss_fn = JointLoss(
    kernel_size=(6, 6, 6),
    weights_kspace_loss=(0.0, 0.0),
    weights_ispace_loss=(0.5, 0.5),
    weights_wavelet_loss=(0.0, 0.0),
    weights_hankel_loss=(0.0, 0.0),
    weights_casorati_loss=(0.0, 0.0),
    normalized_loss=True,
    timing_level=0,
    validation_level=0,
    group=group,
    device=device,
    dtype = dtype
)

### Initializing the optimizer
The optimizer determines which impact the calculated loss has on the models parameters. The impact can be variated via the learning rate lr.

In [6]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    betas=[0.9, 0.999],
    eps=1.0e-8,
    weight_decay=0.0
)

In [7]:
accumulator = GradientAccumulator(
    model,
    accumulation_steps=batch_size_local,
    max_norm=1.0,
    normalized_gradient=False
)

### Initializing the averaged model
The averaged model does not use the models parameters directly it uses the floating average of the model over the last n iterations. Its more robust than using the current average.

In [8]:
averaged_model = LookaheadModel(
    model,
    alpha=0.5,
    k=5
)

## Checkpoint Manager
### Initializing the CheckpointManager
The CheckpointManager is useful for saving and/or loading models. In this case here it either creates a new directory for the model or it loads data from an already existing directory to continue the training at the point where it lastly stopped.

In [9]:
checkpoint_manager = CheckpointManager(
    directory=directory,
    root_dir=root_dir,
    backend=backend,
)

#### Check if the model is already existing and load its last saved state

In [10]:
if load_model_state:
    print("Loading model state ...")
    checkpoint = checkpoint_manager.load(["model_state"], map_location=device)
    if all(checkpoint.values()):
        model.load_state_dict(checkpoint["model_state"])
    else:
        print("Could not load model state.")

Loading model state ...
Could not load model state.


#### Check if the averaged model is already existing and load its last saved state

In [11]:
if load_averaged_model_state:
    print(f"Loading averaged model state ...")
    checkpoint = checkpoint_manager.load(["averaged_model_state"], map_location=device)
    if all(checkpoint.values()):
        averaged_model.load_state_dict(checkpoint["averaged_model_state"])
    else:
        print("Could not load averaged model state.")

Loading averaged model state ...
Could not load averaged model state.


#### Check if the optimizer is already existing and load its last saved state

In [12]:
if load_optim_state:
    print("Loading optim state ...")
    checkpoint = checkpoint_manager.load(["optim_state"], map_location=device)
    if all(checkpoint.values()):
        optimizer.load_state_dict(checkpoint["optim_state"])
    else:
        print("Could not load optim state.")

Loading optim state ...
Could not load optim state.


In [13]:
total_trn_loss = {
    "loss_kspace" : torch.Tensor([]).to(device),
    "loss_ispace" : torch.Tensor([]).to(device),
    "loss_wavelet" : torch.Tensor([]).to(device),
    "loss_hankel" : torch.Tensor([]).to(device),
    "loss_casorati" : torch.Tensor([]).to(device),
    "loss_sum" : torch.Tensor([]).to(device)
}

total_val_loss = list()
iteration = 0

#### Load the last saved state of the loss and the current iteration if existing

In [14]:
if load_metrics:
    print("Loading metrics ...")
    checkpoint = checkpoint_manager.load(["trn_loss", "val_loss", "iteration"])
    if all(checkpoint.values()):
        total_trn_loss = list(checkpoint["trn_loss"])
        total_val_loss = list(checkpoint["val_loss"])
        iteration = checkpoint["iteration"]
    else:
        print("Could not load metrics.")

Loading metrics ...
Could not load metrics.


In [15]:
print(f"Continue with iteration {iteration} ...")

Continue with iteration 0 ...


## Import the data that should be used for the training

In [16]:
h5_path = '/home/jovyan/scripts/juart/examples/dl/ga_kooshball/preprocessing/meas_MID00149_FID31870_Kooshball_GRE_30000_preproc-2.h5'


In [17]:
# Reshape to JUART data format
with h5py.File(h5_path, 'r') as f:
    d = f['us_10/data'][:]
    k = f['us_10/traj'][:]
    C = f['sens_maps'][:]

print("Undersampled Data shape:", d.shape)
print("Undersampled Trajectory shape:", k.shape)
print("Coil sensitivity shape", C.shape)

d = d[0].reshape(-1, d.shape[-1]).transpose(1, 0)
k = k.reshape(3, -1)

d = torch.from_numpy(d)
k = torch.from_numpy(k)
k = k/(torch.max(torch.abs(k)) * 2)

C = np.moveaxis(C, -1, 0)  # coils first
C = torch.from_numpy(C)

print("Undersampled data shape after reshape:", d.shape)
print("Undersampled trajectory shape after reshape:", k.shape)
print("Coil sensitivity shape after reshape", C.shape)


Undersampled Data shape: (1, 128, 2990, 8)
Undersampled Trajectory shape: (3, 128, 2990)
Coil sensitivity shape (128, 128, 128, 8)
Undersampled data shape after reshape: torch.Size([8, 382720])
Undersampled trajectory shape after reshape: torch.Size([3, 382720])
Coil sensitivity shape after reshape torch.Size([8, 128, 128, 128])


### shaping the data
#### In the next steps the data is formed into the right shape so that the training function will understand how the data is structured.

In [18]:
def randomizer(slices, epochs):
    random_list = []

    for e in range(0, epochs, 1):
        random.seed(e)
        random_list.append(random.sample(range(slices), slices))

    return random_list

In [19]:
generator = torch.Generator()
i = -1
random_list = randomizer(nP, num_epochs)

In [20]:
kspace_mask_source = torch.randint(0, 3, (1, d.shape[1]), generator=generator)
kspace_mask_source[kspace_mask_source>1] = 1
kspace_mask_target = 1 - kspace_mask_source

AHd = nonuniform_fourier_transform_adjoint(
        k[:, kspace_mask_source[0]==1],
        d[:, kspace_mask_source[0]==1],
        (nX, nY, nZ)
)
print(AHd.shape)

torch.Size([8, 128, 128, 128])


In [ ]:
while iteration < num_iterations:
    print(f"iteration {iteration} / {num_iterations}")
    tic = time.time()
    generator = torch.Generator()

    if iteration % nP == 0:
        i += 1

    generator.manual_seed(random_list[i][iteration%nP])
    print(f"picked random seed {random_list[i][iteration%nP]}")

    kspace_mask_source = torch.randint(0, 3, (1, d.shape[1], 1, 1), generator=generator)
    kspace_mask_source[kspace_mask_source>1] = 1
    kspace_mask_target = 1 - kspace_mask_source

    d_masked = d * kspace_mask_source
    AHd = nonuniform_fourier_transform_adjoint(
        k[:, kspace_masked_source[0]==1],
        d[:, kspace_masked_source[0]==1],
        (nX, nY, nZ)
    )
    AHd = torch.sum(torch.conj(C) * AHd, dim=0)

    print("Shape of regridded data", AHd.shape)

    data = [
        {
            "images_regridded": AHd,
            "kspace_trajectory": k,
            "sensitivity_maps": C,
            "kspace_mask_source": kspace_mask_source,
            "kspace_mask_target": kspace_mask_target,
            "kspace_data": d,
        }
    ]

    
    # TRAINING
    if model_training:
        print("Define trainings loss...")
        trn_loss = training(
            [0],
            data,
            model,
            loss_fn,
            optimizer,
            accumulator,
            group=group,
            device=device,
        )

        averaged_model.update_parameters(
            model,
        )

        torch.cuda.empty_cache()
        gc.collect()

    else:
        trn_loss = [0] * batch_size

    # VALIDATION
    if model_validation:
        print("Definen validation loss...")
        val_loss = validation(
            validation_index,
            validation_data,
            averaged_model,
            loss_fn,
            group=group,
            device=device,
        )
        torch.cuda.empty_cache()
        gc.collect()

    else:
        val_loss = [0] * batch_size

    for key in total_trn_loss.keys():
        total_trn_loss[key] = torch.cat([total_trn_loss[key], trn_loss[key]])
    
    total_val_loss += val_loss

    # SAVING
    # Completed epoch
    if save_checkpoint and np.mod(iteration + batch_size, nD * nP) == 0:
        print("Creating tagged checkpoint ...")

        checkpoint = {
            "iteration": iteration + batch_size,
            "model_state": model.state_dict(),
            "averaged_model_state": averaged_model.state_dict(),
            "optim_state": optimizer.state_dict(),
            "trn_loss": total_trn_loss,
            "val_loss": total_val_loss,
        }

        epoch = (iteration + batch_size) // (nD * nP)
        checkpoint_manager.save(checkpoint, tag=f"_epoch_{epoch}")

        if single_epoch:
            # Also save the checkpoint as untagged checkpoint
            # Otherwise, training will be stuck in endless loop
            checkpoint_manager.save(checkpoint)
            checkpoint_manager.release()
            break

    # Intermediate checkpoint
    elif save_checkpoint and np.mod(iteration + batch_size, checkpoint_frequency) == 0:
        print("Creating untagged checkpoint ...")

        checkpoint = {
            "iteration": iteration + batch_size,
            "model_state": model.state_dict(),
            "averaged_model_state": averaged_model.state_dict(),
            "optim_state": optimizer.state_dict(),
            "trn_loss": total_trn_loss,
            "val_loss": total_val_loss,
        }

        checkpoint_manager.save(checkpoint, block=False)

    toc = time.time() - tic

    print(
        (
            f"Iteration: {iteration} - "
            + f"Elapsed time: {toc:.0f} - "
            + f"Training loss: {[f'{trn_loss["loss_sum"].item():.3f}']} - "
            + f"Validation loss: {[f'{loss:.3f}' for loss in val_loss]}"
        )
    )

    torch.cuda.empty_cache()
    gc.collect()

    iteration += batch_size

iteration 0 / 30
picked random seed 0
